In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
import tensorflow as tf
import pandas as pd

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import run_unet_mlp_uv15 as base
import run_unet_mlp_bottleneckvqc_uv15_randomsplit as exp

from functions.nb_helpers import (
    signed_log1p,
    transform_y_signed_log1p,
    inverse_transform_y_signed_log1p,
    repo_root,
    resolve_extracted_uv_dir,
    evaluate_split_physical,
    print_region_metrics,
    predict_denorm,
    sample_r2_uv_together,
    plot_uv_threeway,
)

# Back-compat aliases
_sample_r2_uv_together = sample_r2_uv_together

def _predict_denorm(cur_model, X_in, C_in):
    return predict_denorm(cur_model, X_in, C_in, y_mean, y_std, base.MISSING_VALUE)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
height_m = 15.0
last_k = 12
epochs = 1000
batch_size = 2
cond_emb_dim = 16
n_qubits = 5
n_layers = 2
seed = 7
train_frac = 0.8
val_frac = 0.1

base.set_seeds(seed)


In [ ]:
here = str(repo_root())
os.chdir(here)

extracted_uv_dir = str(resolve_extracted_uv_dir(here))

cases = base.list_cases(extracted_uv_dir)

xs, cs, ys, meta = [], [], [], []
for c in cases:
    m_building, u_mean, v_mean = base.load_uv_steady_mean(c.path, height_m=height_m, last_k=last_k)
    x = m_building[..., None].astype(np.float32)
    y = np.stack([u_mean, v_mean], axis=-1).astype(np.float32)
    xs.append(x)
    cs.append(base.cond_vector(c.speed, c.angle_deg))
    ys.append(y)
    meta.append({"file": os.path.basename(c.path), "speed": c.speed, "d_code": c.d_code, "angle_deg": c.angle_deg})

X = np.stack(xs, axis=0)
C = np.stack(cs, axis=0)
Y = np.stack(ys, axis=0)

X.shape, C.shape, Y.shape


In [ ]:
# Non-building u_mean / v_mean distributions
u_all = Y[..., 0]
v_all = Y[..., 1]

valid_mask = (u_all != base.MISSING_VALUE) & (v_all != base.MISSING_VALUE)
u_valid = u_all[valid_mask]
v_valid = v_all[valid_mask]

# Signed log1p transform: x -> sign(x) * log1p(|x|)

u_log = signed_log1p(u_valid)
v_log = signed_log1p(v_valid)

# Train targets in signed-log1p space (buildings keep missing value)
Y_log = transform_y_signed_log1p(Y, base.MISSING_VALUE)

In [ ]:
# wind speed extrapolation
# val -> d03/d04/d07/d08（0/45/180/235）
# test -> d05/d06/d09/d10（90/135/270/315）

train_speeds = {4, 6, 8, 10}
val_test_speed = 12

val_d_codes = {3, 7, 4, 8}
test_d_codes = {5, 9, 6, 10}

train_idx = [i for i, m in enumerate(meta) if m["speed"] in train_speeds]
val_idx = [i for i, m in enumerate(meta) if m["speed"] == val_test_speed and m["d_code"] in val_d_codes]
test_idx = [i for i, m in enumerate(meta) if m["speed"] == val_test_speed and m["d_code"] in test_d_codes]

if len(val_idx) != 4 or len(test_idx) != 4:
    raise ValueError(
        f"Target wind speed {val_test_speed}  has unexpected direction counts: len(val)={len(val_idx)}, len(test)={len(test_idx)}"
    )

remaining = set(range(len(cases))) - set(train_idx) - set(val_idx) - set(test_idx)
if remaining:
    raise ValueError(f"Uncovered samples after split: {sorted(remaining)}")

split_idx = {"train": train_idx, "val": val_idx, "test": test_idx}
tr, va, te = split_idx["train"], split_idx["val"], split_idx["test"]

# Condition scaling (fit on train only)
c_scaler = base.StandardScaler()
C_tr = c_scaler.fit_transform(C[tr])
C_va = c_scaler.transform(C[va])
C_te = c_scaler.transform(C[te])

# Output normalization in signed-log1p space (fit on train only, ignoring missing)
y_mean, y_std = base.compute_y_norm_stats(Y_log[tr])
Y_tr = base.normalize_y(Y_log[tr], y_mean, y_std)
Y_va = base.normalize_y(Y_log[va], y_mean, y_std)
Y_te = base.normalize_y(Y_log[te], y_mean, y_std)

X_tr, X_va, X_te = X[tr], X[va], X[te]
len(tr), len(va), len(te)


# C-QB-UNet

In [ ]:
# # Full epochs training
# model = exp.build_unet_cond_mlp_bottleneck_vqc(
#     input_shape=X_tr.shape[1:],
#     cond_dim=C_tr.shape[-1],
#     cond_emb_dim=cond_emb_dim,
#     n_qubits=n_qubits,
#     n_layers=n_layers,
# )

# model.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history = model.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=epochs,
#     batch_size=batch_size,
#     callbacks=callbacks,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model.load_weights(best_weights_path)


In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_extra12.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model = exp.build_unet_cond_mlp_bottleneck_vqc(
    input_shape=X_tr.shape[1:],
    cond_dim=C_tr.shape[-1],
    cond_emb_dim=cond_emb_dim,
    n_qubits=n_qubits,
    n_layers=n_layers,
)

model.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model.load_weights(best_weights_path)


In [ ]:
print("Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm, pred_tr_denorm, metrics_tr = evaluate_split_physical(model, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm, pred_va_denorm, metrics_va = evaluate_split_physical(model, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm, pred_te_denorm, metrics_te = evaluate_split_physical(model, "test", X_te, C_te, Y_te, y_mean, y_std)

In [ ]:
# Region-wise metrics: top-left open area vs the remaining area
# You can tune these two values to match your "left-top open area" definition.
tl_h, tl_w = 75, 90

# Build region masks from current output size

print("Region-wise metrics (m/s):")
print_region_metrics("train", Y_tr_denorm, pred_tr_denorm, tl_h=tl_h, tl_w=tl_w)
print_region_metrics("val", Y_va_denorm, pred_va_denorm, tl_h=tl_h, tl_w=tl_w)
print_region_metrics("test", Y_te_denorm, pred_te_denorm, tl_h=tl_h, tl_w=tl_w)


# C-UNet

In [ ]:
# # Full epochs training

# # C-UNet config (reuse same data/split)
# mlp_epochs = epochs
# mlp_batch_size = batch_size

# model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])
# model_mlp.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks_mlp = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history_mlp = model_mlp.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=mlp_epochs,
#     batch_size=mlp_batch_size,
#     callbacks=callbacks_mlp,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model_mlp.load_weights(best_weights_path)

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_extra12.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])

model_mlp.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model_mlp.load_weights(best_weights_path)


In [ ]:
print("C-UNet Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm_mlp, pred_tr_denorm_mlp, metrics_tr_mlp = evaluate_split_physical(model_mlp, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm_mlp, pred_va_denorm_mlp, metrics_va_mlp = evaluate_split_physical(model_mlp, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm_mlp, pred_te_denorm_mlp, metrics_te_mlp = evaluate_split_physical(model_mlp, "test", X_te, C_te, Y_te, y_mean, y_std)


In [ ]:
print("Region-wise metrics (m/s):")
print_region_metrics("train", Y_tr_denorm_mlp, pred_tr_denorm_mlp)
print_region_metrics("val", Y_va_denorm_mlp, pred_va_denorm_mlp)
print_region_metrics("test", Y_te_denorm_mlp, pred_te_denorm_mlp)


# U/V Spatial Field

In [ ]:
show_ids = [0, 1, 2]
for i in show_ids:
    m = meta[te[i]]
    plot_uv_threeway(
        Y_te_denorm,
        pred_te_denorm,
        pred_te_denorm_mlp,
        i,
        title=f"U/V Field Comparison ({m['speed']}m/s {m['angle_deg']}deg)",
    )


# Generalization Heatmap

In [ ]:
# Generalization heatmap (R2; one sample per (speed, angle))
# Requires: model, model_mlp, meta, X, C, Y, c_scaler, y_mean, y_std, base, inverse_transform_y_signed_log1p

# ===== 1) Predict all cases (denormalize to physical m/s) =====
C_all = c_scaler.transform(C)

pred_all_vqc = _predict_denorm(model, X, C_all)         # C-QB-UNet
pred_all_mlp = _predict_denorm(model_mlp, X, C_all)     # C-UNet
y_all_true = Y

# ===== 2) Per-sample R2 (U/V pooled, same as evaluate_numpy) =====

rows = []
for i, m in enumerate(meta):
    rows.append({
        "speed": float(m["speed"]),
        "angle_deg": float(m["angle_deg"]),
        "r2_mlp": _sample_r2_uv_together(y_all_true[i], pred_all_mlp[i], base.MISSING_VALUE),
        "r2_vqc": _sample_r2_uv_together(y_all_true[i], pred_all_vqc[i], base.MISSING_VALUE),
    })

df_r2 = pd.DataFrame(rows)

dup = df_r2.duplicated(subset=["speed", "angle_deg"]).any()
pv_mlp = df_r2.pivot(index="angle_deg", columns="speed", values="r2_mlp").sort_index().sort_index(axis=1)
pv_vqc = df_r2.pivot(index="angle_deg", columns="speed", values="r2_vqc").sort_index().sort_index(axis=1)

angles = pv_mlp.index.to_numpy()
speeds = pv_mlp.columns.to_numpy()

# ===== 3) Plot =====
vmin, vmax = 0.8, 1.0
fig, axes = plt.subplots(1, 2, figsize=(6, 3.6), constrained_layout=False)
fig.subplots_adjust(wspace=0.1, right=0.92)  # larger wspace => wider gap between subplots

im0 = axes[0].imshow(pv_mlp.values, origin="lower", aspect="auto", cmap="RdYlGn", vmin=vmin, vmax=vmax)
im1 = axes[1].imshow(pv_vqc.values, origin="lower", aspect="auto", cmap="RdYlGn", vmin=vmin, vmax=vmax)

axes[0].set_title("C-UNet", fontsize=12)
axes[1].set_title("C-QB-UNet", fontsize=12)

for j, ax in enumerate(axes):
    ax.set_xticks(np.arange(len(speeds)))
    ax.set_xticklabels([f"{s:g}" for s in speeds])
    ax.set_xlabel("Wind speed (m/s)", fontsize=10)
    if j == 0:
        ax.set_ylabel("Wind direction (deg)", fontsize=10)
        ax.set_yticks(np.arange(len(angles)))
        ax.set_yticklabels([f"{a:g}" for a in angles], fontsize=10)
    else:
        ax.set_ylabel("")  # Hide y-axis label on the second subplot
        ax.set_yticks([])
        ax.set_yticklabels([])

# Dashed box highlights 12 m/s
highlight_speeds = {12.0}
for hs in highlight_speeds:
    if hs in speeds:
        col = int(np.where(speeds == hs)[0][0])
        rect_kwargs = dict(fill=False, edgecolor="red", linestyle="--", linewidth=2)
        for ax in axes:
            ax.add_patch(Rectangle((col - 0.5, -0.5), 1.0, len(angles), **rect_kwargs))

# colorbar spacing pad=0.05
ticks = np.arange(vmin, vmax + 1e-9, 0.05)
cbar = fig.colorbar(im1, ax=axes.ravel().tolist(), fraction=0.03, pad=0.05, ticks=ticks)
cbar.set_label("R²", fontsize=12)

fig.suptitle("R² Heatmap", fontsize=14)
plt.show()


In [ ]:
# Decay vs. Out-of-Range Degree
# x-axis: wind speed (4, 6, 8, 10, 12)
# y-axis: mean R2 across angles at each speed
# shade: std of R2 across angles

# Reuse df_r2 (columns: speed, angle_deg, r2_mlp, r2_vqc) if available
# Otherwise run the earlier heatmap cell that builds df_r2

speed_order = [4, 6, 8, 10, 12]

# Aggregate by speed (across angles)
agg = (
    df_r2.groupby("speed", as_index=False)
         .agg(
             mean_r2_mlp=("r2_mlp", "mean"),
             std_r2_mlp=("r2_mlp", "std"),
             mean_r2_vqc=("r2_vqc", "mean"),
             std_r2_vqc=("r2_vqc", "std"),
             n_angles=("angle_deg", "nunique"),
         )
)

# Keep x-axis order
agg = agg.set_index("speed").reindex(speed_order).reset_index()

x = agg["speed"].to_numpy(dtype=float)

y_mlp = agg["mean_r2_mlp"].to_numpy(dtype=float)
s_mlp = np.nan_to_num(agg["std_r2_mlp"].to_numpy(dtype=float), nan=0.0)

y_vqc = agg["mean_r2_vqc"].to_numpy(dtype=float)
s_vqc = np.nan_to_num(agg["std_r2_vqc"].to_numpy(dtype=float), nan=0.0)

fig, ax = plt.subplots(1, 1, figsize=(3.7, 3.7))

# C-UNet
ax.plot(x, y_mlp, marker="o", linewidth=2, color="tab:blue", label="C-UNet")
ax.fill_between(x, y_mlp - s_mlp, y_mlp + s_mlp, color="tab:blue", alpha=0.18)

# C-QB-UNet
ax.plot(x, y_vqc, marker="s", linewidth=2, color="tab:orange", label="C-QB-UNet")
ax.fill_between(x, y_vqc - s_vqc, y_vqc + s_vqc, color="tab:orange", alpha=0.18)

ax.set_xticks(speed_order)
ax.set_xlabel("Wind speed (m/s)")
ax.set_ylabel("Mean R² across directions")
ax.set_title("Performance Decay Curves")
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

print(agg)
